# Monster Generator — Simple TensorFlow Version
A small diffusion project for high-school students. **All model code is in this notebook.**

**Run:** Runtime → Change runtime type → GPU, then run the six code cells in order.

**The whole idea:** add noise → learn to predict the noise → start with new noise → remove it repeatedly.

This version keeps just the dataset, a small conditional U-Net, training, and prompt-based generation. The original advanced notebook is still available separately. Training quality depends on the number of steps; the 30-minute A100 goal has not been benchmarked for this TensorFlow version.

## 1. Load the prepared monster images
We download the existing dataset from GitHub. Each image is **32 × 32 pixels with three color channels**.

Each caption has four attributes: **color, eye count, horns, expression**. We use the training split only: **13,200 images**. The other images remain excluded from training.

Our vocabulary has 11 words/concepts. Their IDs are:

| IDs | Concepts |
|---|---|
| 0–3 | green, blue, red, purple |
| 4–6 | one eye, two eyes, three eyes |
| 7–8 | no horns, horns |
| 9–10 | happy, sad |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__, '| GPU:', bool(tf.config.list_physical_devices('GPU')))

# Download the prepared images. No drawing script or extra Python module is needed.
url = 'https://raw.githubusercontent.com/dujing82-blip/tiny-text-to-pixel-ai/main/monster_diffusion/data/monsters32.npz'
path = keras.utils.get_file('monsters32.npz', url,
    file_hash='103c7e0ac28174378fd97f13e413e4a03905b75c3d70e746ea25d99ef3e49a11')
with np.load(path) as data:
    training = data['splits'] == 0
    images = data['images'][training]
    # Convert four separate attribute IDs into our shared 11-word vocabulary.
    word_ids = data['labels'][training].astype('int32') + [0, 4, 7, 9]

# Keep pixels as uint8 until a batch is used, which saves memory.
image_bank = tf.constant(images)
word_bank = tf.constant(word_ids, dtype=tf.int32)
print('Training images:', len(images))

def show_images(pictures, title):
    # The model uses [-1, 1]; convert back to [0, 1] for display.
    pictures = np.clip((np.asarray(pictures) + 1) / 2, 0, 1)
    fig, axes = plt.subplots(1, len(pictures), figsize=(2 * len(pictures), 2.5), squeeze=False)
    for ax, picture in zip(axes[0], pictures):
        ax.imshow(picture, interpolation='nearest')
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_images(images[:6].astype('float32') / 127.5 - 1, 'Actual training examples')

## 2. Add noise to an image
\[
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon.
\]

- $x_0$: clean image, scaled to −1 to 1.
- $\epsilon$: Gaussian noise that **we generate**, so we know its exact values.
- $\bar\alpha_t$: how much signal variance remains at noise level $t$.

We use **200 noise levels**. Larger levels leave less image and more noise. The cosine schedule below makes $\bar\alpha$ decrease smoothly from almost 1 to almost 0. Array indices 0–199 correspond to noisy levels 1–200; the clean image is displayed separately.

In [ ]:
T = 200  # Number of noise levels.

# A cosine curve specifies how much signal remains at each level.
curve = np.cos((np.linspace(0, 1, T + 1) + 0.008) / 1.008 * np.pi / 2) ** 2
curve = curve / curve[0]
beta = np.clip(1 - curve[1:] / curve[:-1], 0, 0.999)
alpha_bar = tf.constant(np.cumprod(1 - beta), dtype=tf.float32)

def add_noise(clean_images, t, noise):
    # One scalar noise level per image; reshape so it broadcasts over pixels.
    a = tf.reshape(tf.gather(alpha_bar, t), [-1, 1, 1, 1])
    return tf.sqrt(a) * clean_images + tf.sqrt(1 - a) * noise

clean = tf.cast(image_bank[:1], tf.float32) / 127.5 - 1
noise = tf.random.normal(tf.shape(clean))
levels = [0, 40, 80, 120, 160, 199]
noisy_pictures = [clean] + [add_noise(clean, [t], noise) for t in levels]
show_images(tf.concat(noisy_pictures, axis=0), 'Clean → increasing noise → almost pure noise')

## 3. Build a small noise-prediction network
The network receives **a noisy image, its noise level, and four word IDs**. It outputs an estimate of the noise, with the same 32 × 32 × 3 shape as the image.

An **Embedding** layer learns 16 numbers for each word. The word vectors and noise level form a condition that enters each convolution block. The U-Net reduces spatial size **32 → 16 → 8**, then expands back to 32. Skip connections carry spatial details from the first half to the second.

In [ ]:
noisy_input = keras.Input((32, 32, 3), name='noisy_image')
time_input = keras.Input((1,), name='noise_level')  # Scaled to [0, 1].
words_input = keras.Input((4,), dtype='int32', name='word_ids')

# Four word IDs → four trainable 16-number vectors → one condition vector.
words = layers.Flatten()(layers.Embedding(11, 16)(words_input))
condition = layers.Dense(64, activation='swish')(
    layers.Concatenate()([words, time_input]))

def conv_block(x, channels):
    x = layers.Conv2D(channels, 3, padding='same', activation='swish')(x)
    # Add the text/time condition to every pixel's feature vector.
    hint = layers.Reshape((1, 1, channels))(layers.Dense(channels)(condition))
    x = layers.Add()([x, hint])
    return layers.Conv2D(channels, 3, padding='same', activation='swish')(x)

# Encoder: gradually reduce the spatial resolution.
a = conv_block(noisy_input, 32)                 # 32 × 32
b = conv_block(layers.AveragePooling2D(2)(a), 64) # 16 × 16
middle = conv_block(layers.AveragePooling2D(2)(b), 96)  # 8 × 8

# Decoder: upsample, then combine with the matching encoder features.
x = layers.Concatenate()([layers.UpSampling2D()(middle), b])
x = conv_block(x, 64)
x = layers.Concatenate()([layers.UpSampling2D()(x), a])
x = conv_block(x, 32)
predicted_noise = layers.Conv2D(3, 3, padding='same')(x)  # Linear output, not sigmoid.

model = keras.Model([noisy_input, time_input, words_input], predicted_noise)
print('Trainable parameters:', model.count_params())

## 4. Train: predict the noise we added
For every batch:
1. Pick clean images and their word IDs.
2. Sample random noise levels and random Gaussian noise.
3. Add the noise, then ask the network to predict it.
4. Minimize the mean squared error between actual and predicted noise.

\[
\mathrm{loss}=\operatorname{mean}[(\epsilon-\hat\epsilon)^2].
\]

Start with **4,000 steps**. Rerunning **this cell** continues training the same model; rerunning Section 3 creates a new model. Reduce the batch size if GPU memory is limited. A lower loss does not guarantee correct eye counts or expressions: inspect generated samples too.

In [ ]:
STEPS = 4000       # Additional batches to train in this cell.
BATCH_SIZE = 64    # Images in each batch.

# Create the optimizer once so rerunning this cell continues its state too.
if 'optimizer_model' not in globals() or optimizer_model is not model:
    optimizer = keras.optimizers.Adam(learning_rate=0.0003, clipnorm=1.0)
    optimizer_model = model

@tf.function  # Compile the training step for speed.
def train_step():
    ids = tf.random.uniform([BATCH_SIZE], 0, len(images), dtype=tf.int32)
    clean = tf.cast(tf.gather(image_bank, ids), tf.float32) / 127.5 - 1
    words = tf.gather(word_bank, ids)
    t = tf.random.uniform([BATCH_SIZE], 0, T, dtype=tf.int32)
    noise = tf.random.normal(tf.shape(clean))
    noisy = add_noise(clean, t, noise)
    time = tf.cast(t[:, None], tf.float32) / (T - 1)
    with tf.GradientTape() as tape:
        prediction = model([noisy, time, words], training=True)
        loss = tf.reduce_mean(tf.square(noise - prediction))
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

losses = []
for step in range(STEPS):
    losses.append(float(train_step()))
    if (step + 1) % 100 == 0:
        print(f'Step {step + 1}/{STEPS} | noise MSE: {np.mean(losses[-100:]):.4f}')

plt.plot(losses)
plt.xlabel('Training step'); plt.ylabel('Noise prediction MSE')
plt.title('Learning to predict the added noise'); plt.show()

## 5. Turn a prompt into four word IDs
Use this controlled vocabulary: **green/blue/red/purple**, **one/two/three eyes**, **horns/no horns**, **happy/sad**.

Example: `happy green monster with three eyes and horns`.

The parser selects attributes; the **trainable embedding layer** converts their IDs into vectors. This tiny project does not understand arbitrary English. If expression or horns are omitted, the defaults are happy and no horns.

In [ ]:
COLORS = ['green', 'blue', 'red', 'purple']
EYES = ['one', 'two', 'three']

def encode_prompt(prompt):
    words = set(prompt.lower().split())
    allowed = set('a an the monster with and eye eyes horns no happy sad'.split() + COLORS + EYES)
    if words - allowed:
        raise ValueError('Unsupported words: ' + ', '.join(sorted(words - allowed)))
    colors = [i for i, color in enumerate(COLORS) if color in words]
    eyes = [i for i, count in enumerate(EYES) if count in words]
    if len(colors) != 1 or len(eyes) != 1:
        raise ValueError('Specify exactly one color and one eye count, e.g. green with two eyes.')
    if {'happy', 'sad'} <= words or ('no' in words and 'horns' not in words):
        raise ValueError('Choose happy OR sad; use "no" only with "horns".')
    horns = int('horns' in words and 'no' not in words)
    sad = int('sad' in words)
    return np.array([colors[0], eyes[0] + 4, horns + 7, sad + 9], dtype='int32')

## 6. Generate monsters from fresh noise
Edit **PROMPT** or **SEED** and rerun this cell. Different seeds give different initial noise.

At each step:
\[
\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}},
\qquad
x_s=\sqrt{\bar\alpha_s}\,\hat x_0+\sqrt{1-\bar\alpha_s}\,\hat\epsilon,
\quad s<t.
\]

First predict the noise, then estimate the clean image, then move to a lower noise level. We use **60 DDIM sampling steps** to make generation faster. The initial noise provides the randomness; this sampler does not add fresh random noise at each step. At the final clean level, $\bar\alpha_s=1$.

The first row shows four generated monsters. The second row shows selected intermediate noisy states for the first monster. **No monster drawing script is used here.**

In [ ]:
PROMPT = 'happy green monster with three eyes and horns'  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}

# The same prompt conditions all four images, but their initial noise differs.
words = tf.constant(np.repeat(encode_prompt(PROMPT)[None], 4, axis=0))
x = tf.random.stateless_normal([4, 32, 32, 3], seed=[SEED, 0])
times = np.linspace(T - 1, 0, 60, dtype=int)
frames = [x[:1]]

# Compile only the neural-network call to keep generation fast and readable.
@tf.function
def predict_noise(x, time, words):
    return model([x, time, words], training=False)

for i, t in enumerate(times):
    time = tf.fill([4, 1], tf.cast(t, tf.float32) / (T - 1))
    noise = predict_noise(x, time, words)
    a = alpha_bar[t]
    clean = (x - tf.sqrt(1 - a) * noise) / tf.sqrt(a)
    clean = tf.clip_by_value(clean, -1, 1)  # Keep estimated pixels in their valid range.
    # Recompute noise after clipping, so the two terms still explain the current x.
    noise = (x - tf.sqrt(a) * clean) / tf.sqrt(1 - a)
    previous_a = alpha_bar[times[i + 1]] if i + 1 < len(times) else tf.constant(1.0)
    x = tf.sqrt(previous_a) * clean + tf.sqrt(1 - previous_a) * noise
    if (i + 1) % 12 == 0:
        frames.append(x[:1])

show_images(x, PROMPT)
show_images(tf.concat(frames, axis=0), 'Reverse diffusion: initial noise → generated monster')

### Try three experiments
1. Keep the prompt fixed and change the seed. Which features stay the same?
2. Keep the seed fixed and change only `green` to `blue`. What changes?
3. If the monsters are unclear, rerun **Section 4** to add more training, then rerun Section 6. Do not recreate the model in Section 3.

**Practical notes:** A GPU is recommended for full training. If you run out of memory, reduce BATCH_SIZE to 32. This minimal notebook keeps its model in the current runtime only; resetting the runtime loses training. It intentionally omits checkpoint management, validation dashboards, guidance controls, and export utilities.

The existing data reserves four full attribute combinations, including the default prompt above. For a familiar training combination, try `happy blue monster with two eyes and horns`. Eye count and expression may be wrong even when the image looks plausible.

References: [DDPM](https://arxiv.org/abs/2006.11239), [DDIM](https://arxiv.org/abs/2010.02502), [TensorFlow training loops](https://www.tensorflow.org/guide/keras/writing_a_training_loop_from_scratch). Original artwork and code follow the repository MIT license.